In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(plotly)
    library(processx)
    library(htmlwidgets)
})

## Combine cds

In [ ]:
cds_nucleolin_1 = readRDS("../03-11-23_F1-E15.25-nucleolin-combined/cds_combined_VCS_JAX_E15-15.5.RDS")

In [ ]:
cds_nucleolin_2 = readRDS("../04-29-23_F1-E15.25-nucleolin-plate45/cds_combined_VCS_JAX_mito_removed_all_umap3.RDS")

In [ ]:
cds_sc35 = readRDS("../04-30-23_F1-E15.25-SC35-plate12/cds_combined_VCS_JAX_mito_removed.RDS")

In [ ]:
cds_sc35_34 = readRDS("../05-06-23_F1-E15.25-SC35-plate34/cds_filtered.RDS")
cds_sc35_56 = readRDS("../05-07-23_F1-E15.25-SC35-plate56/cds_filtered.RDS")

In [ ]:
cds_nuclear_23 = readRDS("../07-23-23_F1-E15.25-nuclear-area-nucleolar-fraction-plate23/cds_filtered.RDS")
cds_nuclear_14 = readRDS("../07-26-23_F1-E15.25-nuclear-area-nucleolar-fraction-plate14/cds_filtered.RDS")
cds_nuclear_5678 = readRDS("../07-28-23_F1-E15.25-nuclear-area-nucleolar-fraction-plate5678/cds_filtered.RDS")

In [ ]:
head(rowData(cds_nuclear_5678))

In [ ]:
rowData_filtered = as.data.frame(rowData(cds_nucleolin_1)) %>%
    filter(gene_ID %in% rowData(cds_sc35)$gene_ID) %>%
    filter(gene_ID %in% rowData(cds_nucleolin_2)$gene_ID) %>%
    filter(gene_ID %in% rowData(cds_sc35_34)$id) %>%
    filter(gene_ID %in% rowData(cds_sc35_56)$id) %>%
    filter(gene_ID %in% rowData(cds_nuclear_23)$id) %>%
    filter(gene_ID %in% rowData(cds_nuclear_14)$id) %>%
    filter(gene_ID %in% rowData(cds_nuclear_5678)$id) 

head(rowData_filtered)
dim(rowData_filtered)

In [ ]:
dim(rowData_filtered)

In [ ]:
cds_nucleolin_2 = cds_nucleolin_2[rownames(rowData_filtered), cds_nuc_sir$Data == "VCS"]
cds_sc35 = cds_sc35[rownames(rowData_filtered), cds_sc35$Data == "VCS"]
cds_nucleolin_1 = cds_nucleolin_1[rownames(rowData_filtered), ]
cds_sc35_34 = cds_sc35_34[rowData(cds_sc35_34)$id %in% rownames(rowData_filtered),]
cds_sc35_56 = cds_sc35_56[rowData(cds_sc35_56)$id %in% rownames(rowData_filtered),]
cds_nuclear_23 = cds_nuclear_23[rowData(cds_nuclear_23)$id %in% rownames(rowData_filtered),]
cds_nuclear_14 = cds_nuclear_14[rowData(cds_nuclear_14)$id %in% rownames(rowData_filtered),]
cds_nuclear_5678 = cds_nuclear_5678[rowData(cds_nuclear_5678)$id %in% rownames(rowData_filtered),]

In [ ]:
cds_sc35_34$Total_RNA = cds_sc35_34$RNA_UMIs
cds_sc35_34$Main_cell_type = NA
cds_sc35_34$Main_trajectory = NA

cds_sc35_56$Total_RNA = cds_sc35_56$RNA_UMIs
cds_sc35_56$Main_cell_type = NA
cds_sc35_56$Main_trajectory = NA

In [ ]:
cds_nuclear_23$Total_RNA = cds_nuclear_23$RNA_UMIs
cds_nuclear_23$Main_cell_type = NA
cds_nuclear_23$Main_trajectory = NA

cds_nuclear_14$Total_RNA = cds_nuclear_14$RNA_UMIs
cds_nuclear_14$Main_cell_type = NA
cds_nuclear_14$Main_trajectory = NA

cds_nuclear_5678$Total_RNA = cds_nuclear_5678$RNA_UMIs
cds_nuclear_5678$Main_cell_type = NA
cds_nuclear_5678$Main_trajectory = NA

In [ ]:
colData(cds_nucleolin_1) = colData(cds_nucleolin_1)[,colnames(colData(cds_nucleolin_1)) %in% colnames(colData(cds_sc35))]
colData(cds_sc35_34) = colData(cds_sc35_34)[,colnames(colData(cds_sc35_34)) %in% colnames(colData(cds_sc35))]
colData(cds_sc35_56) = colData(cds_sc35_56)[,colnames(colData(cds_sc35_56)) %in% colnames(colData(cds_sc35))]

colData(cds_nuclear_23) = colData(cds_nuclear_23)[,colnames(colData(cds_nuclear_23)) %in% colnames(colData(cds_sc35))]
colData(cds_nuclear_14) = colData(cds_nuclear_14)[,colnames(colData(cds_nuclear_14)) %in% colnames(colData(cds_sc35))]
colData(cds_nuclear_5678) = colData(cds_nuclear_5678)[,colnames(colData(cds_nuclear_5678)) %in% colnames(colData(cds_sc35))]



In [ ]:
head(as.data.frame(colData(cds_sc35_56)))

In [ ]:
head(as.data.frame(colData(cds_sc35_34)))

In [ ]:
print(cds_nucleolin_1)
print(cds_nucleolin_2)
print(cds_sc35)
print(cds_sc35_34)
print(cds_sc35_56)
print(cds_nuclear_23)
print(cds_nuclear_14)
print(cds_nuclear_5678)

In [ ]:
cds_nucleolin_1$Experiment = 1
cds_nucleolin_2$Experiment = 2
cds_sc35$Experiment = 3
cds_sc35_34$Experiment = 3
cds_sc35_56$Experiment = 3
cds_nuclear_23$Experiment = 4
cds_nuclear_14$Experiment = 4
cds_nuclear_5678$Experiment = 4

In [ ]:
combine_cds = function(cds_batch_list, min_expr_cutoff = 0, num_cells = 0) {
    combined_mat = NULL
    combined_colData = NULL
    
    for (cds_batch in cds_batch_list) {
        if (is.null(combined_mat)) {
            combined_mat = counts(cds_batch)
            combined_colData = as.data.frame(colData(cds_batch))
        } else {
            combined_mat = cbind(combined_mat, counts(cds_batch))
            combined_colData = rbind(combined_colData, as.data.frame(colData(cds_batch)))
        }
    }
    
    cds = new_cell_data_set(combined_mat, 
                            cell_metadata = combined_colData, 
                            gene_metadata = rowData(cds_batch_list[[1]]))

    cds <- detect_genes(cds, min_expr = min_expr_cutoff)
    expressed_genes <- row.names(subset(cds, num_cells_expressed >= num_cells))
    print(paste0("# of expressed genes: ", length(expressed_genes)))

    cds <- cds[expressed_genes, ]
    cds = estimate_size_factors(cds)
    cds$Total_RNA = Matrix::colSums(counts(cds))
    
    return(cds)
}

In [ ]:
cds_combined = combine_cds(list(cds_nucleolin_1, cds_nucleolin_2, cds_sc35, cds_sc35_34, cds_sc35_56, cds_nuclear_23, cds_nuclear_14, cds_nuclear_5678))
cds_combined

In [ ]:
as.data.frame(colData(cds_combined)) %>%
    count(Data, Experiment)

In [ ]:
head(as.data.frame(colData(cds_combined)))

In [ ]:
saveRDS(cds_combined, "../data/cds_combined_VCS_JAX_mito_removed_all_markers_E15-E15.5.RDS")

## Load cds and run umap

In [ ]:
cds_combined = readRDS("../data/cds_combined_VCS_JAX_mito_removed_all_markers_E15-E15.5.RDS")

In [ ]:
parametricDispersionFit <- function( disp_table, verbose = FALSE, initial_coefs=c(1e-6, 1) )
{
    coefs <- initial_coefs
    iter <- 0
    while(TRUE) {
        residuals <- disp_table$disp / ( coefs[1] + coefs[2] / disp_table$mu )
        good <- disp_table[which(disp_table$disp > 0 & (residuals < 10000) ),]
        if(verbose)
            fit <- glm( disp ~ I(1/mu), data=good,
                        family=Gamma(link="identity"), start=coefs )
        else
            suppressWarnings(fit <- glm( disp ~ I(1/mu), data=good,
                                         family=Gamma(link="identity"), start=coefs ))
        
        oldcoefs <- coefs
        coefs <- coefficients(fit)
        if (coefs[1] < initial_coefs[1]){
            coefs[1] <- initial_coefs[1]
        }
        if (coefs[2] < 0){
            stop( "Parametric dispersion fit failed. Try a local fit and/or a pooled estimation. (See '?estimateDispersions')" )
        }
        if( sum( log( coefs / oldcoefs )^2 ) < coefs[1] )
            break
        iter <- iter + 1

        if( iter > 10 ) {
            warning( "Dispersion fit did not converge." )
            break
        }
    }
    
    if( !all( coefs > 0 ) ){
        stop( "Parametric dispersion fit failed. Try a local fit and/or a pooled estimation. (See '?estimateDispersions')" )
    }
    rm(disp_table)
    
    list(fit, coefs)
}

disp_calc_helper_NB <- function(cds, expressionFamily, min_cells_detected){
    
    rounded <- round(exprs(cds))
    nzGenes <- Matrix::rowSums(rounded > 0.1) 
    nzGenes <- names(nzGenes[nzGenes > min_cells_detected])
    
    x <- DelayedArray(t(t(rounded[nzGenes,]) / size_factors(cds[nzGenes,])))
    
    xim <- mean(1/ size_factors(cds[nzGenes,]))
    
    
    f_expression_mean <- as(DelayedMatrixStats::rowMeans2(x), "sparseVector")
    f_expression_var <- DelayedMatrixStats::rowVars(x)
    
    disp_guess_meth_moments <- f_expression_var - xim * f_expression_mean
    
    disp_guess_meth_moments <- disp_guess_meth_moments / (f_expression_mean^2) #fix the calculation of k
    
    
    res <- data.frame(mu=as.vector(f_expression_mean), disp=as.vector(disp_guess_meth_moments))
    res[res$mu == 0]$mu = NA
    res[res$mu == 0]$disp = NA
    res$disp[res$disp < 0] <- 0
    
    res <- cbind(gene_id=row.names(fData(cds[nzGenes,])), res)
    res
}

estimateDispersionsForCellDataSet <- function(cds, min_cells_detected, removeOutliers, verbose = FALSE)
{
    mu <- NA
    
    progress_opts <- options()$dplyr.show_progress
    options(dplyr.show_progress = T)
    
    cds_pdata <- dplyr::group_by_(dplyr::select_(rownames_to_column(as.data.frame(pData(cds))), "rowname"))
    disp_table <- as.data.frame(disp_calc_helper_NB(cds, cds@expressionFamily, min_cells_detected))

    if(!is.list(disp_table))
        stop("Parametric dispersion fitting failed, please set a different lowerDetectionLimit")

    disp_table <- subset(disp_table, is.na(mu) == FALSE)
    res <- parametricDispersionFit(disp_table, verbose)
    fit <- res[[1]]
    coefs <- res[[2]]

    if (removeOutliers){
        CD <- cooks.distance(fit)
        cooksCutoff <- 4/nrow(disp_table)
        message (paste("Removing", length(CD[CD > cooksCutoff]), "outliers"))
        outliers <- union(names(CD[CD > cooksCutoff]), setdiff(row.names(disp_table), names(CD)))
        res <- parametricDispersionFit(disp_table[row.names(disp_table) %in% outliers == FALSE,], verbose)
        fit <- res[[1]]
        coefs <- res[[2]]
    }
    names( coefs ) <- c( "asymptDisp", "extraPois" )
    ans <- function( q )
        coefs[1] + coefs[2] / q
    attr( ans, "coefficients" ) <- coefs
    
    res <- list(disp_table = disp_table, disp_func = ans)
    return(res)
}
    
dispersionTable <- function (disp_results) 
{
    if (is.null(disp_results)) {
        warning("Warning: estimateDispersions only works, and is only needed, when you're using a CellDataSet with a negbinomial or negbinomial.size expression family")
        stop("Error: no dispersion model found. Please call estimateDispersions() before calling this function")
    }
    disp_df <- data.frame(gene_id = disp_results$disp_table$gene_id, 
                          mean_expression = disp_results$disp_table$mu, 
                          dispersion_fit = disp_results$disp_func(disp_results$disp_table$mu), 
                          dispersion_empirical = disp_results$disp_table$disp)
    return(disp_df)
}

In [ ]:
dispersion_table = estimateDispersionsForCellDataSet(cds_combined, 10000, F)
dispersion_table = dispersionTable(dispersion_table)

In [ ]:
# Get top 3000 highly dispersed genes
disp_table = dispersion_table %>% 
                filter(dispersion_empirical > 0) %>%
                mutate(excess_disp = (dispersion_empirical - dispersion_fit) / dispersion_fit) %>%
                arrange(-excess_disp)

top_subset_genes = as.character(head(disp_table, 2500)$gene_id)


In [ ]:
top_subset_genes = as.data.frame(rowData(cds_combined)) %>% 
    filter(gene_ID %in% top_subset_genes) %>%
    filter(!grepl("^mt", gene_short_name))
top_subset_genes = as.character(top_subset_genes$gene_ID)

In [ ]:
saveRDS(top_subset_genes, "highly_dispersed_genes_3000_E15-E15.5.RDS")

In [ ]:
top_subset_genes = readRDS("highly_dispersed_genes_3000.RDS")

In [ ]:
## filter out mito.genes and subtract mito gene counts
mito_genes <- grep(pattern = "^mt", x = rowData(cds_combined)$gene_short_name, value = TRUE, ignore.case = TRUE)

mito_gene_id = rowData(cds_combined)[rowData(cds_combined)$gene_short_name %in% mito_genes,]$gene_ID
mito_gene_counts = data.frame(Perc_mitochondrial_umis = Matrix::colSums(counts(cds_combined)[mito_gene_id,]))

cds_combined$Perc_mitochondrial_umis = mito_gene_counts / cds_combined$Total_RNA

In [ ]:
summary(cds_combined$Perc_mitochondrial_umis)

In [ ]:
cds_combined = preprocess_cds(cds_combined, num_dim = 30, use_genes = top_subset_genes)
cds_combined = align_cds(cds_combined, alignment_k = 20, alignment_group = "Data", 
                residual_model_formula_str = "~ log(Total_RNA)")

In [ ]:
## Top 2000 variable genes
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=300)
plot_cells(cds_combined, x = 1, y = 2,
           reduction_method = "PCA", color_cells_by = "Data", label_cell_groups = F,
           cell_size = 0.1, alpha = 0.7) + 
        theme(legend.position = "none")

In [ ]:
## Top 2000 variable genes
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=300)
plot_cells(cds_combined, x = 1, y = 2,
           reduction_method = "Aligned", color_cells_by = "Data", label_cell_groups = F,
           cell_size = 0.1, alpha = 0.7) + 
        theme(legend.position = "none")

In [ ]:
cds_combined = reduce_dimension(cds_combined, reduction_method = "UMAP", max_components = 3, cores = 12,
                       umap.n_neighbors = 50, umap.min_dist = 0.1, umap.metric = 'cosine')

In [ ]:
## Top 2000 variable genes
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=300)
plot_cells(cds_combined, x = 1, y = 2,
           reduction_method = "UMAP", color_cells_by = "Data", label_cell_groups = F,
           cell_size = 0.01, alpha = 0.7) + 
        theme(legend.position = "none")

In [ ]:
## Top 2000 variable genes
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=300)
plot_cells(cds_combined, x = 1, y = 3,
           reduction_method = "UMAP", color_cells_by = "Data", label_cell_groups = F,
           cell_size = 0.01, alpha = 0.7) + 
        theme(legend.position = "none")

In [ ]:
## Top 2000 variable genes
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=300)
plot_cells(cds_combined, x = 2, y = 3,
           reduction_method = "UMAP", color_cells_by = "Data", label_cell_groups = F,
           cell_size = 0.01, alpha = 0.7) + 
        theme(legend.position = "none")

In [ ]:
cds_combined = cluster_cells(cds_combined, resolution = 5e-4)
cds_combined$Cluster = clusters(cds_combined)

In [ ]:
## Top 2000 variable genes
options(repr.plot.width=7, repr.plot.height=7, repr.plot.res=300)
plot_cells(cds_combined, x = 1, y = 3,
           reduction_method = "UMAP", color_cells_by = "cluster", label_cell_groups = T,
           cell_size = 0.01, alpha = 0.7) + 
        theme(legend.position = "none")

In [ ]:
saveRDS(cds_combined, "../data/cds_combined_VCS_JAX_mito_removed_all_markers_E15-15.5.RDS")